In [ ]:
import os
import sys
from pathlib import Path

import gc
import torch
from datasets import load_dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import DPOConfig, DPOTrainer

print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())

In [ ]:
# Ensure notebook runs from the repository root
if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Working Directory:", Path.cwd())

DOMAIN = "metformin"

# Base weights source: The Stage 2 instruction model merged previously
stage2_merged_dir = Path(f"./experiments/{DOMAIN}/stage2_instruct/merged_model")

# Stage 3 targets: Where preference alignment artifacts are saved
stage3_dir = Path(f"./experiments/{DOMAIN}/stage3_dpo")
stage3_adapter = stage3_dir / "adapter"
stage3_merged = stage3_dir / "merged_model"
stage3_checkpoints = stage3_dir / "checkpoints"

stage3_dir.mkdir(parents=True, exist_ok=True)
stage3_checkpoints.mkdir(parents=True, exist_ok=True)

assert stage2_merged_dir.exists(), f"Error: Stage 2 merged model not found at {stage2_merged_dir}"

print(f"Base (SFT) Model Path: {stage2_merged_dir.resolve()}")
print(f"Stage 3 Output Dir:     {stage3_dir.resolve()}")

In [ ]:
# Load tokenizer from the Stage 2 merged model
tokenizer = AutoTokenizer.from_pretrained(str(stage2_merged_dir))

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# DPOTrainer strictly requires left-padding for concatenated log-prob evaluation
tokenizer.padding_side = "left"

print(f"Tokenizer vocab size: {len(tokenizer)}")
print(f"Padding side:         {tokenizer.padding_side}")
print(f"Pad Token:            {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")
print(f"EOS Token:            {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")

In [ ]:
dataset_path = Path("data/pharma_preference_data.csv")
if not dataset_path.exists():
    dataset_path = Path("pharma_preference_data.csv")

assert dataset_path.exists(), f"Could not find dataset at {dataset_path.resolve()}"
print(f"Loading preference data from: {dataset_path.resolve()}")

raw_dataset = load_dataset("csv", data_files=str(dataset_path), split="train")

def format_dpo_pairs(example):
    # Support both (instruction, input) and direct prompt schemas
    if "instruction" in example:
        instruction = example["instruction"].strip()
        context = example.get("input", "")
        if context and str(context).strip() and str(context).strip().lower() != "none":
            prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{str(context).strip()}\n\n### Response:\n"
        else:
            prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    else:
        prompt = example["prompt"].strip()
        if not prompt.endswith("### Response:\n"):
            prompt = f"### Instruction:\n{prompt}\n\n### Response:\n"

    # Ensure completions end with the native EOS token
    chosen = example["chosen"].strip() + tokenizer.eos_token
    rejected = example["rejected"].strip() + tokenizer.eos_token

    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

# 90/10 split to monitor preference accuracy during training
dataset_split = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"].map(format_dpo_pairs, remove_columns=raw_dataset.column_names)
eval_dataset = dataset_split["test"].map(format_dpo_pairs, remove_columns=raw_dataset.column_names)

print("=== Sample 0 Preference Pair ===")
print("PROMPT:\n", train_dataset[0]["prompt"])
print("\nCHOSEN:\n", train_dataset[0]["chosen"])
print("\nREJECTED:\n", train_dataset[0]["rejected"])

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading Stage 2 base model in 4-bit from: {stage2_merged_dir}...")
base_model = AutoModelForCausalLM.from_pretrained(
    str(stage2_merged_dir),
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    bias="none",
)

dpo_model = get_peft_model(base_model, lora_config)

# Cast trainable adapter parameters to float32 to prevent PyTorch AMP unscale errors
for param in dpo_model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

dpo_model.print_trainable_parameters()

In [ ]:
use_bf16 = torch.cuda.is_bf16_supported()

dpo_config = DPOConfig(
    output_dir=str(stage3_checkpoints),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,             # DPO uses a lower learning rate than SFT
    lr_scheduler_type="cosine",
    warmup_steps=2,
    beta=0.1,                       # KL penalty strength (typical values: 0.1 to 0.2)
    loss_type="sigmoid",            # Standard Bradley-Terry DPO objective
    max_length=512,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_strategy="steps",
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,
)

trainer = DPOTrainer(
    model=dpo_model,
    ref_model=None,                 # Automatic implicit reference using LoRA disabling
    args=dpo_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting Direct Preference Optimization (DPO)...")
trainer.train()

print(f"Saving final DPO adapter to: {stage3_adapter}...")
trainer.save_model(str(stage3_adapter))
tokenizer.save_pretrained(str(stage3_adapter))

In [ ]:
# Free training structures from GPU memory
del dpo_model, base_model, trainer
gc.collect()
torch.cuda.empty_cache()

print("1. Loading Stage 2 base model in full FP16...")
clean_stage2 = AutoModelForCausalLM.from_pretrained(
    str(stage2_merged_dir),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    device_map="cuda",
)

print(f"2. Loading Stage 3 DPO adapter from: {stage3_adapter}...")
lora_model = PeftModel.from_pretrained(clean_stage2, str(stage3_adapter))

print("3. Fusing DPO weights into base model...")
final_dpo_model = lora_model.merge_and_unload()

print(f"4. Saving final aligned model to: {stage3_merged}...")
final_dpo_model.save_pretrained(str(stage3_merged), safe_serialization=True)
tokenizer.save_pretrained(str(stage3_merged))

print("[SUCCESS] Stage 3 DPO model merged and saved successfully!")

In [ ]:
test_prompt = "### Instruction:\nExplain the mechanism of action of Metformin.\n\n### Response:\n"

inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    outputs = final_dpo_model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

new_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
print("\n--- DPO Aligned Output ---")
print(tokenizer.decode(new_tokens, skip_special_tokens=True).strip())